# RF Vpi Sweep — Scanning Fabry-Perot Cavity

This notebook runs a frequency / power sweep of an RF source while
recording Fabry-Perot cavity transmission traces on an oscilloscope.
The analysis extracts the modulation index $\beta$ from sideband / carrier
area ratios and fits $V_\pi$ at each RF frequency.

## Workflow

1. **Configure** instruments + sweep grid.
2. **(Optional) Power calibration** — uses a **spectrum analyzer** to measure
   the actual RF power at the fundamental frequency for each (frequency, power)
   point.  This avoids the harmonics problem that breaks the scope-based
   sine-fit approach.
3. **Main sweep** — cavity trace acquisition and analysis. Uses calibrated
   voltages when available.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cavityscope.core import SweepConfig, PowerCalibration
from cavityscope.sweep import run_power_calibration, run_sa_power_calibration, run_sweep
from cavityscope.analysis import load_calibration_run, reanalyze_with_calibration

## 1. Instrument initialisation

## Find Instruments

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
print(rm.list_resources('?*'))   # show all VISA resources

for addr in rm.list_resources('?*'):
    try:
        inst = rm.open_resource(addr)
        inst.timeout = 2000
        print(addr, "->", inst.query("*IDN?").strip())
    except Exception as e:
        print(addr, "->", e)

In [ ]:
from hardwarelib.oscilloscopes.rigol import RigolDHO4000
from hardwarelib.signal_generators.windfreak import WindfreakSynthHD
from hardwarelib.spectrum_analyzers.rigol import RigolRSA3000E

SCOPE_RESOURCE  = "USB0::0x1AB1::0x0610::HDO4A243800178::INSTR"
SCOPE_CHANNEL   = 1          # cavity photodetector
SCOPE_TIMEOUT   = 15_000     # ms

RF_COM_PORT     = "COM6"
RF_CHANNEL      = 0
RF_TIMEOUT      = 0.25       # s

SA_RESOURCE     = "USB0::0x1AB1::0x0960::RSA3EXXXXXXXXX::INSTR"  # <-- set your SA VISA address

scope = RigolDHO4000(SCOPE_RESOURCE, timeout_ms=SCOPE_TIMEOUT)
rf    = WindfreakSynthHD(RF_COM_PORT, channel=RF_CHANNEL, timeout_s=RF_TIMEOUT)
sa    = RigolRSA3000E(SA_RESOURCE)

## 2. Sweep configuration

In [ ]:
cfg = SweepConfig(
    rf_frequencies_hz = np.linspace(954e6, 957e6, 21).tolist(),
    rf_powers_dbm     = np.linspace(-20.0, 4, 15).tolist(),

    cavity_fsr_hz         = 10e9,
    carrier_window_hz     = 120e6,
    sideband_window_hz    = 80e6,
    sideband_mode         = "minus",

    compute_vpi           = True,
    net_power_offset_db   = 0.0,
    assumed_load_ohm      = 50.0,

    output_dir            = "vpi_sweep_output",
    save_trace_plots      = True,
    save_reference_plots  = True,
    save_frequency_plots  = True,
    save_raw_traces_csv   = True,

    # --- Spectrum-analyzer power calibration (recommended) ---
    # Uses the RSA3030E to measure the fundamental tone power directly.
    # Immune to harmonics that break the scope-based sine fit.
    cal_use_spectrum_analyzer = True,
    cal_sa_span_hz            = 1e6,    # 1 MHz span around each tone
    cal_sa_rbw_hz             = None,   # auto RBW (or e.g. 3e3 for 3 kHz)
    cal_sa_ref_level_dbm      = 10.0,
    cal_sa_settle_s           = 0.15,
    cal_sa_n_harmonics        = 5,      # measure f, 2f, 3f, 4f, 5f
    cal_sa_save_spectra       = True,   # save wideband spectrum plots per point
    cal_sa_power_offset_db    = 30.0,   # 30 dB external attenuator before SA input
)

## 3. (Optional) Power calibration

Uses the spectrum analyzer to measure the actual RF power at the fundamental
frequency for each (frequency, power) point. The SA cleanly resolves the
fundamental even when the signal generator produces strong harmonics.

Set `cal_use_spectrum_analyzer = False` and `cal_scope_channel = 2` in the
config above to fall back to the legacy scope-based sine-fit method.

In [ ]:
cal = None

if cfg.cal_use_spectrum_analyzer:
    try:
        sa.open()
        rf.open()
        cal = run_sa_power_calibration(sa, rf, cfg)
    finally:
        try:
            rf.set_output(False)
        except Exception:
            pass
        rf.close()
        sa.close()

    print(f"\nCalibration: {cal}")

elif cfg.cal_scope_channel is not None:
    try:
        scope.open()
        rf.open()
        cal = run_power_calibration(scope, rf, cfg)
    finally:
        try:
            rf.set_output(False)
        except Exception:
            pass
        rf.close()
        scope.close()

    print(f"\nCalibration: {cal}")
else:
    print("Calibration skipped.")
    # Uncomment to load a previous calibration CSV instead:
    # cal = PowerCalibration.from_csv("path/to/power_calibration.csv")

In [ ]:
cal_dirs = sorted(Path(cfg.output_dir).glob("calibration_*"))
if cal_dirs:
    cal_run = load_calibration_run(cal_dirs[-1])
    cal_df  = cal_run["cal_df"]
    cal     = cal_run["calibration"]

    freqs = sorted(cal_df['frequency_hz'].unique())
    has_sa_column = 'measured_power_dbm' in cal_df.columns

    if has_sa_column:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        for freq in freqs:
            sub = cal_df[cal_df['frequency_hz'] == freq].sort_values('power_dbm')
            ax1.plot(sub['power_dbm'], sub['measured_power_dbm'], 'o-', ms=4, lw=1.2,
                     label=f"{freq/1e9:.4f} GHz")
            ax2.plot(sub['power_dbm'], sub['vpk_v'], 'o-', ms=4, lw=1.2,
                     label=f"{freq/1e9:.4f} GHz")
        ax1.plot(ax1.get_xlim(), ax1.get_xlim(), 'k--', lw=0.8, alpha=0.4)
        ax1.set(xlabel='RF power setting (dBm)', ylabel='Measured power (dBm)',
                title='SA power calibration — dBm')
        ax1.grid(True, alpha=0.25)
        ax1.legend(fontsize=7, ncol=max(1, len(freqs)//5))
        ax2.set(xlabel='RF power setting (dBm)', ylabel='Vpk (V)',
                title='SA power calibration — Vpk')
        ax2.grid(True, alpha=0.25)
        ax2.legend(fontsize=7, ncol=max(1, len(freqs)//5))
    else:
        fig, ax2 = plt.subplots(figsize=(9, 4))
        for freq in freqs:
            sub = cal_df[cal_df['frequency_hz'] == freq].sort_values('power_dbm')
            ax2.plot(sub['power_dbm'], sub['vpk_v'], 'o-', ms=4, lw=1.2,
                     label=f"{freq/1e9:.4f} GHz")
        ax2.set(xlabel='RF power setting (dBm)', ylabel='Measured Vpk (V)',
                title='Scope power calibration')
        ax2.grid(True, alpha=0.25)
        ax2.legend(fontsize=7, ncol=max(1, len(freqs)//5))

    plt.tight_layout()
    plt.show()
else:
    print("No calibration runs found.")

## 3b. Harmonic analysis

The signal generator (especially after amplification) produces harmonics at
$2f$, $3f$, $4f$, … in addition to the fundamental at $f$.  This section
analyses the harmonic content measured by the spectrum analyzer during
calibration.

### Why this matters for $V_\pi$ estimation

The electro-optic modulator responds to the voltage at its electrodes.
The modulation index is $\beta = \pi V_\mathrm{pk}(f) / V_\pi$, where
$V_\mathrm{pk}(f)$ is the peak voltage **at the fundamental frequency only**.

- **Scope-based calibration fails** when the signal has strong harmonics:
  the sine fit tries to match a distorted waveform and either over- or
  under-estimates the fundamental amplitude.
- **SA-based calibration** measures the power in the fundamental tone
  directly (the SA resolves each harmonic into its own spectral line),
  so the resulting $V_\mathrm{pk}$ is correct regardless of distortion.

The harmonic analysis below quantifies:
- **THD (%)** — total harmonic distortion as a fraction of fundamental voltage.
  High THD means the amplifier is being driven nonlinearly.
- **dBc levels** — each harmonic's power relative to the fundamental.
  Harmonics below −30 dBc are negligible; above −20 dBc they indicate
  significant nonlinearity.
- **Power fraction** — what percentage of the total output power is actually
  in the fundamental.  This is what the modulator sees.

> **Practical guidance:** If THD stays below ~5% across your sweep, the
> amplifier is linear enough that even the scope-based method would likely
> work (but the SA method is still better).  If THD exceeds ~10–20%, the
> SA calibration is essential — the scope fit would give wrong voltages.

In [ ]:
from cavityscope.analysis import (
    build_harmonics_dataframe, build_thd_dataframe,
    plot_thd_summary, plot_harmonic_waterfall, plot_harmonic_heatmap,
)
from IPython.display import display, Image as IPImage

cal_dirs = sorted(Path(cfg.output_dir).glob("calibration_*"))
if cal_dirs:
    cal_dir = cal_dirs[-1]

    # --- THD summary ---
    thd_csv = cal_dir / "thd_summary.csv"
    if thd_csv.exists():
        thd_df = pd.read_csv(thd_csv)
        print(f"THD summary: {len(thd_df)} measurement points")
        print(f"  Mean THD:  {thd_df['thd_percent'].mean():.1f}%")
        print(f"  Max  THD:  {thd_df['thd_percent'].max():.1f}%")
        print(f"  Min fundamental fraction: {thd_df['fundamental_power_fraction'].min()*100:.1f}%")
        print()

        thd_png = cal_dir / "thd_summary.png"
        if thd_png.exists():
            display(IPImage(filename=str(thd_png), width=900))
    else:
        print("No THD data found (harmonic analysis was not run).")

    # --- Harmonic heatmap ---
    heatmap_png = cal_dir / "harmonic_heatmap.png"
    if heatmap_png.exists():
        display(IPImage(filename=str(heatmap_png), width=700))

    # --- Per-frequency harmonic waterfall ---
    waterfall_pngs = sorted(cal_dir.glob("harmonic_waterfall_*.png"))
    if waterfall_pngs:
        print(f"\n{'='*60}\nHarmonic waterfall plots ({len(waterfall_pngs)} frequencies)\n{'='*60}")
        for p in waterfall_pngs:
            display(IPImage(filename=str(p), width=900))

    # --- Browse individual spectrum plots (show a few at highest power) ---
    spectra_dir = cal_dir / "spectra"
    if spectra_dir.exists():
        all_spectra = sorted(spectra_dir.glob("*.png"))
        max_pwr = max(cfg.rf_powers_dbm)
        pwr_str = f"{max_pwr:+06.2f}dBm"
        top_spectra = [p for p in all_spectra if pwr_str in p.name]
        if not top_spectra:
            top_spectra = all_spectra[-3:]  # fallback: last 3
        print(f"\n{'='*60}\nExample spectra at highest power ({len(top_spectra)} shown)\n{'='*60}")
        for p in top_spectra:
            display(IPImage(filename=str(p), width=900))
else:
    print("No calibration runs found.")

In [ ]:
if cal_dirs:
    harm_csv = cal_dirs[-1] / "harmonics.csv"
    if harm_csv.exists():
        harm_df = pd.read_csv(harm_csv)

        # Show dBc levels at highest power for each frequency
        max_pwr = harm_df["power_dbm"].max()
        at_max = harm_df[harm_df["power_dbm"] == max_pwr].copy()
        pivot = at_max.pivot_table(
            index="frequency_hz", columns="harmonic_number",
            values="level_dbc",
        )
        pivot.columns = [f"{'fund' if c == 1 else f'{int(c)}f'} (dBc)" for c in pivot.columns]
        pivot.index = [f"{f/1e9:.4f} GHz" for f in pivot.index]
        print(f"Harmonic levels (dBc) at P_set = {max_pwr:+.1f} dBm:\n")
        display(pivot.style.format("{:.1f}").background_gradient(
            cmap="RdYlGn", vmin=-60, vmax=0, axis=None))
    else:
        print("No harmonics.csv found.")

## 4. Main sweep

The scope timebase should be set for the cavity trace before running this.
If a calibration was measured above, it is passed in automatically.

In [ ]:
try:
    scope.open()
    rf.open()

    data = run_sweep(
        scope=scope,
        rf_source=rf,
        cfg=cfg,
        scope_channel=SCOPE_CHANNEL,
        calibration=cal,
    )
finally:
    try:
        rf.set_output(False)
    except Exception:
        pass
    rf.close()
    scope.close()

## 5. Inspect results

In [ ]:
df_results = data["results"]
df_fits    = data["fits"]

print(f"{len(df_results)} measurement points, {len(df_fits)} frequency fits")
if 'voltage_source' in df_results.columns:
    print(f"Voltage source: {df_results['voltage_source'].iloc[0]}")
df_fits

In [ ]:
if not df_fits.empty:
    freq_ghz = df_fits["rf_frequency_hz"] / 1e9
    vpi = df_fits["fit_vpi_v"]
    r2 = df_fits["fit_r2"]
    n_pts = df_fits["n_fit_points"]
    valid = np.isfinite(vpi)

    fig, (ax_vpi, ax_r2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    ax_vpi.plot(freq_ghz[valid], vpi[valid], "o-", markersize=7, lw=1.5, label="$V_\\pi$")
    for f, v, n in zip(freq_ghz[valid], vpi[valid], n_pts[valid]):
        ax_vpi.annotate(f"N={int(n)}", (f, v), textcoords="offset points",
                        xytext=(0, 8), fontsize=7, ha="center", alpha=0.7)
    if valid.any():
        idx_min = int(np.nanargmin(vpi[valid]))
        f_min = float(freq_ghz[valid].iloc[idx_min] if hasattr(freq_ghz[valid], 'iloc') else freq_ghz[valid][idx_min])
        v_min = float(vpi[valid].iloc[idx_min] if hasattr(vpi[valid], 'iloc') else vpi[valid][idx_min])
        ax_vpi.axvline(f_min, ls=":", lw=0.8, color="tab:red", alpha=0.6)
        ax_vpi.annotate(
            f"min $V_\\pi$ = {v_min:.3f} V\n@ {f_min*1e3:.2f} MHz",
            (f_min, v_min),
            textcoords="offset points", xytext=(12, -18),
            fontsize=8, color="tab:red", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="tab:red", lw=1.2),
        )
    ax_vpi.set_ylabel("$V_\\pi$ (V)")
    ax_vpi.set_title("$V_\\pi$ vs RF frequency")
    ax_vpi.grid(True, alpha=0.25)
    ax_vpi.legend()

    bar_w = 0.8 * float(np.min(np.diff(freq_ghz))) if len(freq_ghz) > 1 else 0.1
    ax_r2.bar(freq_ghz[valid], r2[valid], width=bar_w, alpha=0.7)
    ax_r2.set_ylim(0, 1.05)
    ax_r2.axhline(0.99, ls=":", lw=0.8, color="gray", alpha=0.5, label="R\u00b2=0.99")
    ax_r2.set(xlabel="RF frequency (GHz)", ylabel="Fit R\u00b2")
    ax_r2.grid(True, alpha=0.25)
    ax_r2.legend()

    plt.tight_layout()
    plt.show()

## 5b. (Optional) Re-analyze with a different power calibration

Use this to apply a calibration **after** the sweep has already been recorded,
or to swap in a new calibration CSV. Only the voltage mapping and Vpi fits
are recomputed — the raw sideband/carrier measurements stay unchanged.

In [ ]:
# --- Option A: load calibration from a saved run folder ---
# cal_run  = load_calibration_run("vpi_sweep_output/calibration_YYYYMMDD_HHMMSS")
# cal_post = cal_run["calibration"]

# --- Option B: load the most recent calibration run ---
# cal_run  = load_calibration_run(sorted(Path(cfg.output_dir).glob("calibration_*"))[-1])
# cal_post = cal_run["calibration"]

# --- Option C: load a bare CSV ---
# cal_post = PowerCalibration.from_csv("path/to/power_calibration.csv")

# --- Option D: reload sweep results from disk ---
# df_results = pd.read_csv("vpi_sweep_output/measurement_.../sweep_results.csv")

# Recompute with calibration (set calibration=None for analytical dBm→V)
data_recal = reanalyze_with_calibration(
    results_df=df_results,
    cfg=cfg,
    calibration=cal,         # swap in cal_post or None as needed
    output_dir=None,         # set a path to save updated CSVs and plots
)
df_results = data_recal["results"]
df_fits    = data_recal["fits"]
df_fits

## 6. Browse saved plots

In [ ]:
from IPython.display import display, Image as IPImage

run_dir = sorted(Path(cfg.output_dir).glob("measurement_*"))[-1]

def show_plots(folder: Path, heading: str):
    pngs = sorted(folder.glob("*.png"))
    if not pngs:
        return
    print(f"\n{'='*60}\n{heading}  ({len(pngs)} plots)\n{'='*60}")
    for p in pngs:
        display(IPImage(filename=str(p), width=900))

for name, label in [
    ("vpi_vs_frequency.png", "Vpi vs frequency summary"),
    ("power_calibration.png", "Scope power calibration"),
]:
    p = Path(cfg.output_dir) / name
    if not p.exists():
        p = run_dir / name
    if p.exists():
        print(f"\n{'='*60}\n{label}\n{'='*60}")
        display(IPImage(filename=str(p), width=900))

show_plots(run_dir / "reference_plots", "Reference traces (time domain)")
show_plots(run_dir / "trace_plots",     "RF-on traces (time domain)")
show_plots(run_dir / "frequency_plots", "Traces in frequency space")
show_plots(run_dir / "fit_plots",       "Per-frequency Vpi fits")